[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gamaly/dpgs-analysis-2026/blob/main/02_teams_and_activity.ipynb)


In [1]:
# ======================================================================================================
#  SETUP (inserted by the build script). In Google Colab this downloads the repository so that the
#  `data/` folder sits next to the notebook; on your own computer it does nothing at all.
# ======================================================================================================
import os, sys

REPO_SLUG = "gamaly/dpgs-analysis-2026"     # <github user>/<repository>; set in scripts/99b_build_public_repo.py
BRANCH = "main"

IN_COLAB = "google.colab" in sys.modules      # Colab imports this module in every session; a local Jupyter does not

if IN_COLAB and not os.path.isdir("data"):
    # Colab starts with an empty working directory, so the data folder is missing. Fetch the whole repository
    # (a few tens of MB): first try a shallow `git clone`, and if that fails download the branch zip instead.
    import subprocess, urllib.request, zipfile
    repo_dir = REPO_SLUG.split("/")[-1]
    try:
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://github.com/{REPO_SLUG}.git", repo_dir], check=True)
    except Exception as err:
        print("git clone failed (", err, ") -> downloading the zip instead")
        url = f"https://github.com/{REPO_SLUG}/archive/refs/heads/{BRANCH}.zip"
        urllib.request.urlretrieve(url, "repo.zip")
        with zipfile.ZipFile("repo.zip") as z:
            z.extractall(".")                                   # unpacks to <repository>-<branch>/
        repo_dir = f"{repo_dir}-{BRANCH}"
    os.chdir(repo_dir)                                         # from here on, "data/" and "extras/" resolve as they do locally
    print("Colab: working in", os.getcwd())
    # Colab already has pandas, numpy and matplotlib. Install only what is missing (nothing, normally).
    import importlib.util
    missing = [p for p in ("pandas", "numpy", "matplotlib") if importlib.util.find_spec(p) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
elif not IN_COLAB:
    # Local use: run Jupyter from the repository folder (the one containing this notebook and data/).
    assert os.path.isdir("data"), "Start Jupyter in the repository folder: 'data/' must sit next to this notebook."


# 02. Teams and activity: core contributors, output per core contributor, ageing, features, newcomers

**What this notebook does.** It rebuilds the post's central tables from the DPG-quarter panel:
how many people do *sustained* work on each DPG, how much each of them produces, whether that differs for old and
new projects, how many "feature" PRs and releases there are, and how many first-time contributors appear.
It reproduces `scripts/13_findings_tables.py` (copy in `pipeline/`) from the released data.

**Key definitions.**
* **Core contributor** (a core contributor is a person active in at least 2 of the 3 months of a quarter): a person active (authored a commit or merged PR, or submitted a review)
  in at least **2 of the 3 months** of a quarter, de-duplicated across all of a DPG's repos, bots excluded.
  It is a proxy for "people doing sustained work", which paid developers almost always are. It is *not* a head count
  of staff: squash merges credit only the PR author, non-code roles are invisible.
* **Sample.** The 166 high-confidence DPGs (notebook 01), fixed cohort of repos (rates use a fixed set of repositories created before 2024; the fixed set of repositories is those created before 2024 or holding at least 50 earlier commits), the partial current
  quarter (2026Q4) dropped. 2026 therefore means January to September (three quarters).
* **Balanced panel.** The 100 DPGs that have panel rows in *every* quarter from 2021Q1 to 2026Q3. Using the same
  DPGs in every period means changes cannot come from DPGs entering the data. It is a subset of long-lived projects.
* **Output per core contributor** is a ratio of totals (sum of output / sum of core contributors), never an
  average of per-DPG ratios. Output A = minor + major releases + issues closed (what AI does not inflate);
  output B = merged human PRs (AI can inflate it; team size and output are never built from the same events: never read as causal next to a commit-based team).
* **Releases** are classified by version number against the highest seen so far (`release_first` / `release_backport`
  are "unclassified"); tag-only repos produce most unclassified ones.

**Inputs:** `dpg_quarterly.csv`, `repo_quarterly.csv`, `people_quarterly.csv` (hashed person ids), `dpg_confidence.csv`.

In [2]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
DATA = Path("data")
# Each "Claim Nxxx" cell below stores the number(s) it reproduces with record_result(...); the last cell of the
# notebook prints them all with print_results(). (Nothing is read from or written to disk.)
RESULTS = {}

def record_result(claim_id, description, values=(), ok=None, note=""):
    # values: the numbers the cell reproduced; ok: a yes/no test for claims that are not a number
    RESULTS[claim_id] = {"description": description, "values": [float(v) for v in values], "ok": ok, "note": note}

def print_results():
    # plain summary: claim id, what it is, the numbers (or the yes/no test)
    for cid, r in RESULTS.items():
        shown = ", ".join(f"{v:.6g}" for v in r["values"]) if r["values"] else ("yes" if r["ok"] else "no")
        print(f"{cid}  {r['description']}\n      -> {shown}" + (f"   ({r['note']})" if r["note"] else ""))
    print(f"\n{len(RESULTS)} claims reproduced in this notebook")

## 1. Load and select the sample

In [3]:
dq = pd.read_csv(DATA / "dpg_quarterly.csv")
rq = pd.read_csv(DATA / "repo_quarterly.csv")
conf = pd.read_csv(DATA / "dpg_confidence.csv")

high = set(conf.loc[conf.dpg_confidence == "high", "dpgid"])            # the 166 DPGs

# The collection date falls inside 2026Q4, so that quarter is flagged partial. Drop every partial quarter.
partial_q = set(dq.loc[dq.partial_quarter.astype(str) == "True", "quarter"]) | \
            set(rq.loc[rq.partial_quarter.astype(str) == "True", "quarter"])
d = dq[dq.dpgid.isin(high) & ~dq.quarter.isin(partial_q)].copy()
d["year"] = d.quarter.str[:4].astype(int)
last_q = d.quarter.max()
print("partial quarters dropped:", sorted(partial_q), "| last complete quarter:", last_q)
print("DPGs with panel rows:", d.dpgid.nunique(), "of", len(high))

# Output definitions
d["output_a"] = d.release_minor + d.release_major + d.issues_closed         # primary output
d["output_b"] = d.pr_merged_human                                           # merged human PRs (secondary)
d["output_c"] = d.release_total + d.issues_closed                           # all releases + issues closed
d["unclassified"] = d.release_total - d.release_major - d.release_minor - d.release_patch

# Project age: creation year of the DPG's oldest cohort repo (used for the ageing split, section 5)
rc = rq[rq.in_cohort.astype(str) == "True"].copy()
rc["created"] = pd.to_datetime(rc.repo_created, errors="coerce")
oldest = rc.groupby("dpgid").created.min()
d["oldest_repo_created"] = d.dpgid.map(oldest)
d["age_group"] = pd.NA
d.loc[d.oldest_repo_created < "2018-01-01", "age_group"] = "created_before_2018"
d.loc[(d.oldest_repo_created >= "2018-01-01") & (d.oldest_repo_created < "2024-01-01"), "age_group"] = "created_2018_2023"

partial quarters dropped: ['2026Q4'] | last complete quarter: 2026Q3
DPGs with panel rows: 144 of 166


## 2. The balanced panel
DPGs that have a panel row in every complete quarter from 2021Q1. (2020 is skipped: data start in 2020Q1, so many
DPGs "appear" then simply because collection began.)

In [4]:
def balanced_ids(start):
    need = sorted(q for q in d.quarter.unique() if q >= start)
    cnt = d[d.quarter.isin(need)].groupby("dpgid").quarter.nunique()
    return set(cnt[cnt == len(need)].index), need[0]

bal_ids, start_q = balanced_ids("2021Q1")
db = d[d.dpgid.isin(bal_ids) & (d.quarter >= start_q)]
print(f"balanced panel: {len(bal_ids)} DPGs, quarters {start_q}..{last_q}")

balanced panel: 100 DPGs, quarters 2021Q1..2026Q3


## 3. Table builders (same logic as `13_findings_tables.py`)

In [5]:
def d_groupby(g, key, agg):
    keys = key if isinstance(key, list) else [key]
    rows = []
    for k, x in g.groupby(keys):
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(keys, k)), **agg(x).to_dict()})
    return pd.DataFrame(rows)

def core_table(g, key):
    """Core contributors per period: total, median per DPG, and the share of DPGs with 0 / 1-2 / 3+."""
    def agg(x):
        c = x.core_team_dpg
        return pd.Series({
            "n_dpgs": x.dpgid.nunique(), "n_dpg_quarters": len(x), "n_quarters": x.quarter.nunique(),
            "core_total": c.sum(), "core_median_per_dpg": c.median(),
            "share_dpgs_core_0": (c == 0).mean(), "share_dpgs_core_1_2": c.between(1, 2).mean(),
            "share_dpgs_core_3plus": (c >= 3).mean(),
            "active_total": x.active_contributors_dpg.sum(), "reviewers_total": x.reviewers_dpg.sum()})
    return d_groupby(g, key, agg)

def ratio_table(g, key):
    """Output per core contributor as a ratio of totals."""
    def agg(x):
        core = x.core_team_dpg.sum()
        rt = x.release_total.sum()
        return pd.Series({
            "n_dpgs": x.dpgid.nunique(), "n_dpg_quarters": len(x), "n_quarters": x.quarter.nunique(),
            "core_total": core, "output_a_total": x.output_a.sum(), "output_b_total": x.output_b.sum(),
            "output_c_total": x.output_c.sum(), "release_total": rt, "unclassified_releases": x.unclassified.sum(),
            "unclassified_share": x.unclassified.sum() / rt if rt else float("nan"),
            "output_a_per_core": x.output_a.sum() / core if core else float("nan"),
            "output_c_per_core": x.output_c.sum() / core if core else float("nan"),
            "output_b_per_core": x.output_b.sum() / core if core else float("nan")})
    return d_groupby(g, key, agg)

t1 = core_table(db, "year")                       # balanced, by year
t2 = ratio_table(db, "year")
t3 = core_table(db[db.age_group.notna()], ["age_group", "year"])
print(t1.round(3).to_string(index=False))

 year  n_dpgs  n_dpg_quarters  n_quarters  core_total  core_median_per_dpg  share_dpgs_core_0  share_dpgs_core_1_2  share_dpgs_core_3plus  active_total  reviewers_total
 2021   100.0           400.0         4.0      3221.0                  5.0              0.125                0.190                  0.685        6059.0           2843.0
 2022   100.0           400.0         4.0      3335.0                  5.0              0.115                0.210                  0.675        6238.0           3262.0
 2023   100.0           400.0         4.0      3503.0                  5.0              0.150                0.192                  0.658        6347.0           3534.0
 2024   100.0           400.0         4.0      3379.0                  5.0              0.142                0.198                  0.660        5975.0           3567.0
 2025   100.0           400.0         4.0      3321.0                  4.0              0.228                0.158                  0.615        5846.0    

## 4. Claims: the balanced panel, core contributors and output per person

### Claim N033: the same 100 DPGs throughout

In [6]:
record_result("N033", "DPGs in the balanced panel", [len(bal_ids)])
print(len(bal_ids), "DPGs; n_dpgs by year:", t1.n_dpgs.unique())

100 DPGs; n_dpgs by year: [100.]


### Claim N048: core contributors per quarter, summed over the 100 DPGs
Year totals are sums of four quarterly counts (person-quarters); dividing by the number of quarters gives the
average per quarter. 2026 has three quarters.

In [7]:
t1y = t1.set_index("year")
core_pq = t1y.core_total / t1y.n_quarters
print(core_pq.round(1).to_string())
record_result("N048", "core contributors per quarter 2021 / 2023 / 2025 / 2026",
         [core_pq[2021], core_pq[2023], core_pq[2025], core_pq[2026]])

year
2021    805.2
2022    833.8
2023    875.8
2024    844.8
2025    830.2
2026    831.7


### Claim N049: share of DPG-quarters with no core contributor (2021 vs 2026)

In [8]:
print((t1y.share_dpgs_core_0 * 100).round(1).to_string())
record_result("N049", "share of DPG-quarters with 0 core contributors (%), 2021 and 2026",
         [t1y.share_dpgs_core_0[2021] * 100, t1y.share_dpgs_core_0[2026] * 100])

year
2021    12.5
2022    11.5
2023    15.0
2024    14.2
2025    22.8
2026    24.0


### Claim N050: median core contributors per DPG, 2021 vs 2026

In [9]:
print(t1y.core_median_per_dpg.to_string())
record_result("N050", "median core contributors per DPG-quarter, 2021 and 2026",
         [t1y.core_median_per_dpg[2021], t1y.core_median_per_dpg[2026]])

year
2021    5.0
2022    5.0
2023    5.0
2024    5.0
2025    4.0
2026    4.0


### Claim N039: merged PRs per core contributor (output B)
Reminder (team size and output are never built from the same events; review and role measures are descriptive: more PRs is not a sign of a healthier product): *more PRs is not a sign of a healthier product*. This is shown next to output A below.

In [10]:
t2y = t2.set_index("year")
print(t2y[["core_total", "output_b_total", "output_b_per_core"]].round(2).to_string())
b = t2y.output_b_per_core
record_result("N039", "merged PRs per core contributor 2021 / 2024 / 2025 / 2026 and rise 2021->2026 (%)",
         [b[2021], b[2024], b[2025], b[2026], (b[2026] / b[2021] - 1) * 100])

      core_total  output_b_total  output_b_per_core
year                                               
2021      3221.0         28199.0               8.75
2022      3335.0         30203.0               9.06
2023      3503.0         36101.0              10.31
2024      3379.0         36383.0              10.77
2025      3321.0         36646.0              11.03
2026      2495.0         36825.0              14.76


### Claim N040: releases (minor + major) plus issues closed per core contributor (output A)

In [11]:
a = t2y.output_a_per_core
print(a.round(2).to_string())
record_result("N040", "output A per core contributor 2021 / 2024 / 2025 / 2026", [a[2021], a[2024], a[2025], a[2026]])

year
2021    7.39
2022    6.61
2023    6.12
2024    6.59
2025    5.72
2026    6.18


## 5. Claims: old versus new projects (ageing check)
If the thinning of teams were just ageing, the *older* projects would shrink most. The split uses the creation year
of each DPG's oldest cohort repo.

### Claim N051: the 52 DPGs created 2018-2023

In [12]:
t3i = t3.set_index(["age_group", "year"])
g = "created_2018_2023"
print(t3i.loc[g][["n_dpgs", "core_median_per_dpg", "share_dpgs_core_0"]])
record_result("N051", "2018-23 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026",
         [t3i.loc[(g, 2021), "n_dpgs"], t3i.loc[(g, 2021), "core_median_per_dpg"], t3i.loc[(g, 2026), "core_median_per_dpg"],
          t3i.loc[(g, 2021), "share_dpgs_core_0"] * 100, t3i.loc[(g, 2026), "share_dpgs_core_0"] * 100])

      n_dpgs  core_median_per_dpg  share_dpgs_core_0
year                                                
2021    52.0                  3.0           0.187500
2022    52.0                  3.0           0.182692
2023    52.0                  3.0           0.206731
2024    52.0                  3.0           0.206731
2025    52.0                  2.0           0.322115
2026    52.0                  1.5           0.352564


### Claim N052: the 48 older DPGs (created before 2018)

In [13]:
g = "created_before_2018"
print(t3i.loc[g][["n_dpgs", "core_median_per_dpg", "share_dpgs_core_0"]])
record_result("N052", "pre-2018 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026",
         [t3i.loc[(g, 2021), "n_dpgs"], t3i.loc[(g, 2021), "core_median_per_dpg"], t3i.loc[(g, 2026), "core_median_per_dpg"],
          t3i.loc[(g, 2021), "share_dpgs_core_0"] * 100, t3i.loc[(g, 2026), "share_dpgs_core_0"] * 100])

      n_dpgs  core_median_per_dpg  share_dpgs_core_0
year                                                
2021    48.0                  7.0           0.057292
2022    48.0                  8.0           0.041667
2023    48.0                  8.5           0.088542
2024    48.0                  8.0           0.072917
2025    48.0                  7.0           0.125000
2026    48.0                  6.0           0.118056


## 6. Claims: feature PRs and releases ("weak measures")
`pr_feature_labelled_or_titled` counts merged human PRs that a team *itself* labelled or titled as a feature, so it
depends on each team's labelling habits. Releases: minor + major releases ("feature releases"). The unclassified
share shows how much of the release stream the version-number method cannot classify.

### Claim N034: feature PRs, 2021 vs first nine months of 2026

In [14]:
by_year = db.groupby("year")
feat = by_year.pr_feature_labelled_or_titled.sum()
print(feat.to_string())
record_result("N034", "feature-labelled merged PRs 2021 and 2026", [feat[2021], feat[2026]])

year
2021    3121
2022    2914
2023    3565
2024    4095
2025    4780
2026    5247


### Claim N035: feature PRs as a share of merged human PRs

In [15]:
share = by_year.pr_feature_labelled_or_titled.sum() / by_year.pr_merged_human.sum() * 100
print(share.round(2).to_string())
record_result("N035", "feature share of merged PRs (%), 2021 and 2026", [share[2021], share[2026]])

year
2021    11.07
2022     9.65
2023     9.88
2024    11.26
2025    13.04
2026    14.25


### Claim N036: minor and major releases per year

In [16]:
feat_rel = by_year.release_minor.sum() + by_year.release_major.sum()
print(feat_rel.to_string())
record_result("N036", "minor+major releases 2021 and 2026", [feat_rel[2021], feat_rel[2026]])

year
2021    423
2022    405
2023    413
2024    611
2025    896
2026    913


### Claim N037: DPGs shipping at least one feature release in the year

In [17]:
db2 = db.assign(fr=db.release_minor + db.release_major)
dpgs_fr = db2[db2.fr > 0].groupby("year").dpgid.nunique()      # distinct DPGs with a minor/major release that year
print(dpgs_fr.to_string())
record_result("N037", "DPGs with a feature release: 2021, 2024, 2025, 2026",
         [dpgs_fr[2021], dpgs_fr[2024], dpgs_fr[2025], dpgs_fr[2026]])

year
2021    49
2022    52
2023    50
2024    60
2025    55
2026    55


### Claim N038: share of releases the method could not classify

In [18]:
print((t2y.unclassified_share * 100).round(1).to_string())
record_result("N038", "unclassified release share (%), 2021 and 2026",
         [t2y.unclassified_share[2021] * 100, t2y.unclassified_share[2026] * 100])

year
2021    61.5
2022    51.1
2023    61.3
2024    53.5
2025    45.6
2026    32.7


## 7. Claim: who writes, reviews and merges (unbalanced sample)
Among DPG-quarters with at least one human merged PR: median number of authors, reviewers and mergers.
This table uses all high-confidence DPGs with rows (not the balanced panel), as in `t5`.

### Claim N053: median about 9 writers, 5 reviewers, 3 mergers per DPG-quarter

In [19]:
m = d[d.pr_merged_human > 0]
t5 = m.groupby("year")[["authors_dpg", "reviewers_dpg", "mergers_dpg"]].median()
print(t5.to_string())
span = m[(m.year >= 2021) & (m.year <= 2025)]                     # the post says "between 2021 and 2025"
med = span[["authors_dpg", "reviewers_dpg", "mergers_dpg"]].median()
print("median over 2021-2025 DPG-quarters:", med.to_dict())
record_result("N053", "median authors / reviewers / mergers per DPG-quarter, 2021-2025", list(med))

      authors_dpg  reviewers_dpg  mergers_dpg
year                                         
2020         11.0            7.0          3.5
2021         10.0            5.0          3.0
2022          9.0            5.0          3.0
2023          9.0            5.0          3.0
2024          8.0            5.0          3.0
2025          9.0            5.0          3.0
2026         11.0            5.0          3.0
median over 2021-2025 DPG-quarters: {'authors_dpg': 9.0, 'reviewers_dpg': 5.0, 'mergers_dpg': 3.0}


## 8. First-time contributors
Built from `people_quarterly.csv`: one row per person per DPG per quarter, with a **hashed** person id
(so no one can be named). A person's first quarter in a DPG is their first appearance; they count once as a
first-timer there. *First-timers are new to that DPG, not new to open source.*
* 2020 is a **burn-in** year (everyone active then looks new), so trends start in 2021.
* **Retention** = active in the same DPG in any of the next 4 quarters; **later core** = core in any later quarter.
  Both need a full 4-quarter follow-up, so recent cohorts are censored and their shares are left blank.

In [20]:
pq = pd.read_csv(DATA / "people_quarterly.csv")
pq = pq[pq.dpgid.isin(high) & ~pq.quarter.isin(partial_q)].copy()
pq["year"] = pq.quarter.str[:4].astype(int)
pq["qi"] = pq.quarter.str[:4].astype(int) * 4 + pq.quarter.str[5].astype(int) - 1      # quarter index
last_qi = int(last_q[:4]) * 4 + int(last_q[5]) - 1
pq["core"] = pq.core.astype(str) == "True"
pq = pq.sort_values(["dpgid", "person_id", "qi"])
key = ["dpgid", "person_id"]
first = pq.groupby(key).head(1).copy()                                 # each person's first quarter in each DPG
later = pq.merge(first[key + ["qi"]].rename(columns={"qi": "fqi"}), on=key)
later = later[later.qi > later.fqi]
back4 = later[later.qi <= later.fqi + 4].groupby(key).size().rename("n_back4")
core_later = later[later.core].groupby(key).size().rename("n_core_later")
first = first.join(back4, on=key).join(core_later, on=key)
first["evaluable"] = first.qi <= last_qi - 4                           # a full follow-up window exists
first["returned"] = first.n_back4.fillna(0) > 0
first["became_core"] = first.n_core_later.fillna(0) > 0

def first_table_year(ids, start):
    f = first[first.dpgid.isin(ids) & (first.quarter >= start)]
    dd = d[d.dpgid.isin(ids) & (d.quarter >= start)]
    rows = []
    for yr, x in f.groupby("year"):
        ev = x[x.evaluable]
        npd = dd.loc[dd.year == yr, "dpgid"].nunique()
        rows.append({"year": yr, "first_timers": len(x), "first_timers_per_dpg": len(x) / npd,
                     "n_evaluable": len(ev),
                     "share_returned_within_4q": ev.returned.mean() if len(ev) else np.nan,
                     "share_ever_core_later": ev.became_core.mean() if len(ev) else np.nan})
    return pd.DataFrame(rows).set_index("year")

ft = first_table_year(bal_ids, start_q)
print(ft.round(4).to_string())

      first_timers  first_timers_per_dpg  n_evaluable  share_returned_within_4q  share_ever_core_later
year                                                                                                  
2021          1876                 18.76         1876                    0.3705                 0.2457
2022          1926                 19.26         1926                    0.3458                 0.2269
2023          1710                 17.10         1710                    0.3263                 0.2158
2024          1394                 13.94         1394                    0.3874                 0.2188
2025          1244                 12.44          928                    0.3621                 0.1918
2026          1359                 13.59            0                       NaN                    NaN


### Claim N067: first-time contributors per DPG, 2021-2025

In [21]:
fpd = ft.first_timers_per_dpg
record_result("N067", "first-timers per DPG 2021..2025", [fpd[y] for y in range(2021, 2026)])
print(fpd.round(2).to_string())

year
2021    18.76
2022    19.26
2023    17.10
2024    13.94
2025    12.44
2026    13.59


### Claim N068: nine months of 2026, and the annualised figure
The annualised figure multiplies by 4/3 (three quarters to four). This is simple arithmetic, not a forecast.

In [22]:
print(round(fpd[2026], 2), "-> annualised", round(fpd[2026] * 4 / 3, 2), "(from the rounded 13.69:", round(13.69 * 4 / 3, 2), ")")
record_result("N068", "first-timers per DPG in 2026 (Jan-Sep) and annualised", [fpd[2026], fpd[2026] * 4 / 3])

13.59 -> annualised 18.12 (from the rounded 13.69: 18.25 )


### Claim N069: share of newcomers active again within a year (2021-2025)

In [23]:
ret = ft.share_returned_within_4q.loc[2021:2025] * 100
print(ret.round(1).to_string())
record_result("N069", "range of share returning within 4 quarters (%), 2021-2025", [ret.min(), ret.max()])

year
2021    37.0
2022    34.6
2023    32.6
2024    38.7
2025    36.2


### Claim N070: share of newcomers who later become core (2021 vs 2025)

In [24]:
corel = ft.share_ever_core_later * 100
print(corel.round(1).to_string())
record_result("N070", "share later core (%), 2021 and 2025", [corel[2021], corel[2025]])

year
2021    24.6
2022    22.7
2023    21.6
2024    21.9
2025    19.2
2026     NaN


### Claim N098: both 2025 newcomer figures rest on 934 of 1,252 newcomers with a full year of follow-up
A 2025 newcomer needs four further quarters to be judged on "came back" or "became core"; only those who arrived early
enough (`n_evaluable`) are counted in the 2025 shares. The 2025 values are therefore partly censored.

In [25]:
print(ft.loc[2025, ["first_timers", "n_evaluable"]].to_dict())
record_result("N098", "2025 newcomers with a full follow-up year, of all 2025 newcomers", [ft.loc[2025, "n_evaluable"], ft.loc[2025, "first_timers"]])

{'first_timers': 1244, 'n_evaluable': 928}


## 10. More claims from the panel (v7): issues, concentration, ageing, newcomers

The cells below reuse the objects built above (`db` = balanced panel, `d`, `bal_ids`, `first`, `t3`, `oldest`).

### Claim N103: issues closed are about 95% of the second output measure
Output A is minor + major releases + issues closed. Its fall per person is therefore mostly a fall in issues closed per
person. The share is issues closed divided by output A, summed over the 100 balanced DPGs, by year.

In [26]:
iss = db.groupby("year")[["issues_closed", "release_minor", "release_major"]].sum()
iss_share = iss.issues_closed / (iss.issues_closed + iss.release_minor + iss.release_major) * 100
print(iss_share.round(1).to_string())
print("issues closed 2021 and 2026 (Jan-Sep):", int(iss.issues_closed[2021]), int(iss.issues_closed[2026]))
record_result("N103", "issues closed as % of output A, 2021 and 2026", [iss_share[2021], iss_share[2026]])

year
2021    98.2
2022    98.2
2023    98.1
2024    97.3
2025    95.3
2026    94.1
issues closed 2021 and 2026 (Jan-Sep): 23395 14498


### Claim N124: issues opened per core contributor fell too
Sum of issues opened divided by sum of core contributors, balanced panel, by year.

In [27]:
iop = db.groupby("year").issues_opened.sum() / db.groupby("year").core_team_dpg.sum()
print(iop.round(2).to_string())
record_result("N124", "issues opened per core contributor, 2021 and 2026", [iop[2021], iop[2026]])

year
2021    7.54
2022    7.58
2023    6.67
2024    5.97
2025    5.77
2026    6.37


### Claim N104: the rise in merged PRs per person is concentrated in about twenty fast-growing projects
Rank the 100 balanced DPGs by how much their merged human PRs per quarter grew (2026 per quarter minus 2021 per quarter;
2026 has three quarters, 2021 four). Drop the 20 fastest and recompute the ratio of totals. Then look at each project
separately: among the DPGs that had at least one core contributor in both years, how many saw PRs per core contributor rise,
and what is the median ratio 2026 / 2021?

In [28]:
yr_pr = db.groupby(["dpgid", "year"]).pr_merged_human.sum().unstack()
growth = ((yr_pr[2026] / 3) - (yr_pr[2021] / 4)).sort_values(ascending=False)     # PRs per quarter, 2026 minus 2021

def pr_per_core_without(n):
    # ratio of totals for the balanced panel minus the n DPGs with the largest growth (n = 0 keeps everyone)
    x = db[~db.dpgid.isin(growth.head(n).index)]
    g = x.groupby("year").agg(pr=("pr_merged_human", "sum"), core=("core_team_dpg", "sum"))
    return g.pr / g.core

for n in (0, 5, 10, 20):
    r = pr_per_core_without(n)
    print(f"minus top {n:2d}: 2021 {r[2021]:.2f} -> 2026 {r[2026]:.2f}")
r20 = pr_per_core_without(20)

# Project by project: only DPG-years with at least one core contributor have a ratio
z = db.groupby(["dpgid", "year"]).agg(pr=("pr_merged_human", "sum"), core=("core_team_dpg", "sum")).reset_index()
z = z[z.core > 0]
z["ratio"] = z.pr / z.core
w = z.pivot(index="dpgid", columns="year", values="ratio").dropna(subset=[2021, 2026])
median_ratio = (w[2026] / w[2021]).median()
share_rose = (w[2026] > w[2021]).mean() * 100
print(f"{len(w)} DPGs with core > 0 in both years | median ratio 2026/2021 = {median_ratio:.2f} | share that rose = {share_rose:.0f}%")
record_result("N104", "PRs per core, 2021 and 2026 without the 20 fastest growers; 20; median project ratio; % of projects that rose; n projects",
         [r20[2021], r20[2026], 20, median_ratio, share_rose, len(w)])

minus top  0: 2021 8.75 -> 2026 14.76
minus top  5: 2021 8.49 -> 2026 12.62
minus top 10: 2021 8.87 -> 2026 11.85
minus top 20: 2021 9.17 -> 2026 9.99


82 DPGs with core > 0 in both years | median ratio 2026/2021 = 1.57 | share that rose = 63%


### Claim N106: the younger group has more sustained people than in 2021, the older group the same
Core contributors per quarter (sum over DPGs divided by the number of quarters in the year) in the two age groups of the
balanced panel: oldest repository created 2018-2023, and before 2018. This reuses table `t3` from section 3.

In [29]:
t3i = t3.set_index(["age_group", "year"])
per_q = (t3i.core_total / t3i.n_quarters).unstack("year")           # rows: age group, columns: year
print(per_q[[2021, 2023, 2025, 2026]].round(1).to_string())
newer = per_q.loc["created_2018_2023"]; older = per_q.loc["created_before_2018"]
record_result("N106", "core per quarter, newer group 2021 and 2026, % change; older group 2021 and 2026",
         [newer[2021], newer[2026], (newer[2026] / newer[2021] - 1) * 100, older[2021], older[2026]])

year                  2021   2023   2025   2026
age_group                                      
created_2018_2023    239.5  286.8  268.8  268.3
created_before_2018  565.8  589.0  561.5  563.3


### Claim N107: panel presence, and how many of the newer DPGs have a GitHub repository created in 2021 or later
A DPG is in the balanced panel if it has a quarterly row in every complete quarter from 2021Q1. Thirteen of the 52 newer
DPGs have an oldest repository **created on GitHub** in 2021 or later and still have rows from 2021: their history was
moved to GitHub (or the repository is a successor), so the repository's creation date is later than the project's start.

In [30]:
old_year = oldest.reindex(list(bal_ids)).dt.year                       # creation year of each balanced DPG's oldest cohort repo
newer_ids = db.loc[db.age_group == "created_2018_2023", "dpgid"].unique()
late = old_year.loc[newer_ids]; late = late[late >= 2021]
print("oldest-repo creation year, balanced DPGs:", old_year.value_counts().sort_index().to_dict())
print(f"{len(late)} of {len(newer_ids)} newer DPGs have an oldest repo created in 2021 or later:", late.value_counts().sort_index().to_dict())
record_result("N107", "newer DPGs with oldest repo created 2021+; newer DPGs; created in 2021, 2022, 2023; (the years 2021 and 2022 as written in the post)",
         [len(late), len(newer_ids), int((late == 2021).sum()), int((late == 2022).sum()), int((late == 2023).sum()), 2021, 2022])

oldest-repo creation year, balanced DPGs: {2009: 1, 2011: 5, 2012: 3, 2013: 6, 2014: 6, 2015: 9, 2016: 11, 2017: 7, 2018: 10, 2019: 9, 2020: 20, 2021: 10, 2022: 1, 2023: 2}
13 of 52 newer DPGs have an oldest repo created in 2021 or later: {2021: 10, 2022: 1, 2023: 2}


### Claim N109: first-time contributors in the first nine months of each year
2026 has only three complete quarters, so each year is compared over the **same window, January to September (Q1-Q3)**.
`first` (section 8) holds each person's first quarter in each DPG; the count is divided by the 100 balanced DPGs.

In [31]:
fb = first[first.dpgid.isin(bal_ids)]                       # first appearances in the balanced panel's DPGs
n_bal = len(bal_ids)
def first_q1q3(y):
    # people whose first quarter in a DPG falls in Q1-Q3 of year y
    return fb[(fb.quarter >= f"{y}Q1") & (fb.quarter <= f"{y}Q3")]
same_window = {y: len(first_q1q3(y)) / n_bal for y in range(2021, 2027)}
print({y: round(v, 2) for y, v in same_window.items()})
print("2026 against 2025: +%.0f%%; against 2021: %.0f%%" % ((same_window[2026] / same_window[2025] - 1) * 100,
                                                          (same_window[2026] / same_window[2021] - 1) * 100))
record_result("N109", "first-timers per DPG in Q1-Q3 of 2021..2026", [same_window[y] for y in range(2021, 2027)])

{2021: 14.37, 2022: 13.35, 2023: 12.98, 2024: 10.26, 2025: 9.28, 2026: 13.59}
2026 against 2025: +46%; against 2021: -5%


### Claim N110: five large projects supply most of the rise in newcomers
Compare Q1-Q3 2026 with Q1-Q3 2025 for each DPG: the increase in first-time contributors. How much of the total increase
comes from the five DPGs with the largest increase? What does the median DPG show? And what is the mean without the five DPGs
that have the most first-timers over the whole period?

In [32]:
a25 = first_q1q3(2025).groupby("dpgid").size().reindex(list(bal_ids), fill_value=0)
a26 = first_q1q3(2026).groupby("dpgid").size().reindex(list(bal_ids), fill_value=0)
inc = (a26 - a25).sort_values(ascending=False)                # increase per DPG
total_inc, top5_inc = int(inc.sum()), int(inc.head(5).sum())
print(f"increase {total_inc}; the five largest supply {top5_inc} ({top5_inc / total_inc:.0%})")
print("median DPG, Q1-Q3 2025 and 2026:", a25.median(), a26.median())
top5_overall = fb.groupby("dpgid").size().sort_values(ascending=False).head(5).index       # most first-timers over all years
m25, m26 = a25[~a25.index.isin(top5_overall)].mean(), a26[~a26.index.isin(top5_overall)].mean()
print(f"mean without the five DPGs with most first-timers overall: {m25:.1f} -> {m26:.1f}")
record_result("N110", "increase in first-timers explained by five DPGs, of the total, %; median DPG 2025 and 2026; five; mean without the five, 2025 and 2026",
         [top5_inc, total_inc, top5_inc / total_inc * 100, a25.median(), a26.median(), 5, m25, m26])

increase 431; the five largest supply 259 (60%)


median DPG, Q1-Q3 2025 and 2026: 3.0 3.0
mean without the five DPGs with most first-timers overall: 7.4 -> 11.0


### Claim N111: first-time contributors per project fell by a third between 2021 and 2025
Full years, balanced panel (table `ft` from section 8). The fall is the ratio of the 2025 to the 2021 figure.

In [33]:
fall = (ft.first_timers_per_dpg[2025] / ft.first_timers_per_dpg[2021] - 1) * 100
print(round(ft.first_timers_per_dpg[2021], 2), "->", round(ft.first_timers_per_dpg[2025], 2), "=", round(fall, 1), "%")
record_result("N111", "change in first-timers per DPG, 2021 to 2025 (%)", [fall])

18.76 -> 12.44 = -33.7 %


## 11. The slice by Sustainable Development Goal (SDG)
Each DPG lists the SDGs it serves in the registry; **a DPG is counted under every goal it lists**, so the groups overlap.
Sample: the 166 high-confidence DPGs. Per goal: DPGs, dormant DPGs (notebook 01), DPGs with a merged AI-feature PR since
2024 (notebook 03), and the change in the sum of core contributors per quarter between 2023 and the first three quarters
of 2026 (for DPGs with data in both periods). This reproduces `pipeline/13c_sdg_slice.py` from released files.

In [34]:
import re

# --- DPG -> SDG list (the registry text repeats per DPG-repo row; keep one row per DPG) --------------------------------------
rmap2 = pd.read_csv(DATA / "dpg_repo_map.csv")[["dpgid", "sdgs"]].drop_duplicates("dpgid")
sd = pd.DataFrame([(r.dpgid, int(n)) for r in rmap2.itertuples() for n in re.findall(r"SDG\s*(\d+)", str(r.sdgs))],
                  columns=["dpgid", "sdg"]).drop_duplicates()

# --- status, AI-feature work, core contributors ------------------------------------------------------------------------------
st2 = pd.read_csv(DATA / "dpg_status.csv")
st2 = st2[(st2.repo_set == "high") & st2.dpgid.isin(high)][["dpgid", "status"]]
ai_pr = pd.read_csv(DATA / "ai_feature_prs.csv")
ai_dpgs = set(ai_pr[(ai_pr.confidence == "high") & (ai_pr.merged_at >= "2024")].dpgid)       # AI-feature PR merged since 2024
c23 = dq[dq.quarter.between("2023Q1", "2023Q4")].groupby("dpgid").core_team_dpg.mean()      # mean core per quarter, 2023
c26 = dq[dq.quarter.between("2026Q1", "2026Q3")].groupby("dpgid").core_team_dpg.mean()      # 2026Q4 is partial: excluded
both = c23.index.intersection(c26.index).intersection(list(high))                      # sample DPGs with data in both periods
nm = dq.drop_duplicates("dpgid").set_index("dpgid").dpg                                       # DPG names

sdg_rows = []
dsd = sd[sd.dpgid.isin(high)].merge(st2, on="dpgid", how="left")
for k, g in dsd.groupby("sdg"):
    ids = set(g.dpgid); bi = both.intersection(list(ids))
    a_, b_ = c23[bi].sum(), c26[bi].sum()
    sdg_rows.append({"sdg": k, "dpgs": len(ids), "dormant": int((g.status == "dormant").sum()),
                     "dormant_share": round((g.status == "dormant").mean(), 3),
                     "ai_since_2024": len(ids & ai_dpgs), "ai_share": round(len(ids & ai_dpgs) / len(ids), 3),
                     "n_both": len(bi), "core_2023": round(a_, 1), "core_2026": round(b_, 1),
                     "core_change": round(b_ / a_ - 1, 3) if a_ else None})
t9 = pd.DataFrame(sdg_rows).set_index("sdg").sort_values("dpgs", ascending=False)
print(t9.to_string())

def grp(ids):
    # change in summed core contributors between the two periods for a set of DPGs, and the median project change
    ids = [i for i in ids if i in both]; a_, b_ = c23[ids], c26[ids]
    return dict(n=len(ids), a=a_.sum(), b=b_.sum(), chg=(b_.sum() / a_.sum() - 1) * 100,
                median_chg=((b_ / a_)[a_ > 0] - 1).median() * 100)

h3 = sd[(sd.sdg == 3) & sd.dpgid.isin(high)].dpgid.tolist()      # health DPGs in the sample
h16 = sd[(sd.sdg == 16) & sd.dpgid.isin(high)].dpgid.tolist()    # governance DPGs in the sample

     dpgs  dormant  dormant_share  ai_since_2024  ai_share  n_both  core_2023  core_2026  core_change
sdg                                                                                                  
3      60       19          0.317             14     0.233      54      274.7      202.3       -0.263
9      52       10          0.192             22     0.423      43      393.7      390.0       -0.009
16     44        3          0.068             23     0.523      39      290.5      292.0        0.005
4      32        9          0.281             14     0.438      30      210.9      189.3       -0.102
17     27        9          0.333              9     0.333      26      183.9      167.3       -0.090
11     24        4          0.167             10     0.417      22      103.8       99.3       -0.043
10     24        4          0.167             10     0.417      23      156.8      167.3        0.068
8      24        5          0.208              8     0.333      20      147.2     

### Claim N116: health, the largest group, is thinning fastest
SDG 3 (good health and well-being): number of DPGs, how many are dormant, how many have built AI features since 2024, and the
change in sustained contributors, plus the median project's change.

In [35]:
r3 = t9.loc[3]; g3 = grp(h3)
print(r3.to_dict()); print("median project change: %.1f%%" % g3["median_chg"])
record_result("N116", "SDG3: DPGs; dormant, share %; AI since 2024, share %; DPGs with both periods; core 2023 and 2026; change %; median project change %",
         [r3.dpgs, r3.dormant, r3.dormant_share * 100, r3.ai_since_2024, r3.ai_share * 100, r3.n_both, r3.core_2023, r3.core_2026,
          r3.core_change * 100, g3["median_chg"]])

{'dpgs': 60.0, 'dormant': 19.0, 'dormant_share': 0.317, 'ai_since_2024': 14.0, 'ai_share': 0.233, 'n_both': 54.0, 'core_2023': 274.7, 'core_2026': 202.3, 'core_change': -0.263}
median project change: -48.2%


### Claim N117: governance software (SDG 16) is holding
Same figures for SDG 16 (peace, justice and strong institutions). The +0.5% in the sum is within noise: the median project
changed by -4.5%.

In [36]:
r16 = t9.loc[16]; g16 = grp(h16)
print(r16.to_dict()); print("median project change: %.1f%%" % g16["median_chg"])
record_result("N117", "SDG16: DPGs; dormant, share %; AI since 2024, share %; DPGs with both periods; core 2023 and 2026; change %; median project change %",
         [r16.dpgs, r16.dormant, r16.dormant_share * 100, r16.ai_since_2024, r16.ai_share * 100, r16.n_both, r16.core_2023, r16.core_2026,
          r16.core_change * 100, g16["median_chg"]])

{'dpgs': 44.0, 'dormant': 3.0, 'dormant_share': 0.068, 'ai_since_2024': 23.0, 'ai_share': 0.523, 'n_both': 39.0, 'core_2023': 290.5, 'core_2026': 292.0, 'core_change': 0.005}
median project change: -4.5%


### Claim N118: three projects account for about two-thirds of health's loss
Within SDG 3, the three biggest decliners (OpenSRP, CommCare, OpenEyes) lost 47 of the 72 core contributors that the group lost in
total. The group's change is then recomputed (a) without those three, (b) without the three largest by 2023 size, and (c)
leaving out each of the six biggest decliners in turn. Finally a bootstrap (5,000 resamples of the DPGs, **seed 0**) gives a
95% interval for the group's change. The bootstrap uses `numpy.random.default_rng(0)`; the same seed and numpy version give the
same interval; with another numpy version the interval can move by a tenth of a point.

In [37]:
in3 = [i for i in h3 if i in both]
t3x = pd.DataFrame({"dpg": nm[in3], "c23": c23[in3], "c26": c26[in3]}); t3x["d"] = t3x.c26 - t3x.c23     # per-DPG change
dec = t3x.sort_values("d")                                      # biggest decliners first
print(dec.head(6).round(1).to_string())
top3_dec = dec.index[:3]; top3_size = t3x.sort_values("c23", ascending=False).index[:3]
tot_delta = t3x.d.sum(); top3_delta = dec.d.head(3).sum()
no_dec = grp([i for i in h3 if i not in top3_dec]); no_size = grp([i for i in h3 if i not in top3_size])
loo = [grp([j for j in h3 if j != i])["chg"] for i in dec.index[:6]]       # leave one of the six biggest decliners out
print(f"total change {tot_delta:.1f}; top-3 decliners {top3_delta:.1f} ({top3_delta / tot_delta:.0%}); without them {no_dec['chg']:.1f}%;"
      f" without the 3 largest by size {no_size['chg']:.1f}%; leave-one-out {min(loo):.1f}% to {max(loo):.1f}%")

def sdg3_bootstrap(rng):
    # 5,000 resamples (with replacement) of the SDG3 DPGs; change in the summed core contributors of each resample
    a_, b_ = c23[in3].to_numpy(), c26[in3].to_numpy()
    return [(b_[k].sum() / a_[k].sum() - 1) for k in (rng.integers(0, len(a_), len(a_)) for _ in range(5000))]
rng0 = np.random.default_rng(0)                                  # FIXED seed
boot = sdg3_bootstrap(rng0)
ci = np.percentile(boot, [2.5, 97.5]) * 100
print("bootstrap 95%% interval: %.1f%% to %.1f%%" % tuple(ci))
v = [dec.c23.iloc[k] for k in range(3)] + [dec.c26.iloc[k] for k in range(3)]
record_result("N118", "OpenSRP, CommCare, OpenEyes 2023 -> 2026 core; lost by the three, of the group's total, %; change without them; 3; change without the 3 largest; leave-one-out range; bootstrap interval",
         [dec.c23.iloc[0], dec.c26.iloc[0], dec.c23.iloc[1], dec.c26.iloc[1], dec.c23.iloc[2], dec.c26.iloc[2],
          -top3_delta, -tot_delta, top3_delta / tot_delta * 100, no_dec["chg"], 3, no_size["chg"], min(loo), max(loo), ci[0], ci[1]],
         ok=set(nm[list(top3_dec)]) == {"OpenSRP", "CommCare", "OpenEyes"})

                                                                      dpg   c23   c26     d
dpgid                                                                                      
GID0090085                                                        OpenSRP  21.0   1.7 -19.3
GID0090016                                                       CommCare  30.2  15.7 -14.6
GID0090257                                                       OpenEyes  13.5   0.0 -13.5
GID0090110  Surveillance Outbreak Response Management and Analysis System  13.0   5.3  -7.7
GID0091592                                                          JemPI   5.5   0.0  -5.5
GID0090100                                                         Simple   9.0   3.7  -5.3
total change -72.3; top-3 decliners -47.4 (66%); without them -11.9%; without the 3 largest by size -17.4%; leave-one-out -25.2% to -20.9%
bootstrap 95% interval: -45.7% to -3.8%


### Claim N125: ranking of all 17 groups; only one smaller group lost a larger share than health
Change in summed sustained contributors 2023 to 2026 (Q1-Q3) for every SDG group, smallest first, with the number of DPGs
and the number with data in both periods. Zero hunger (12 DPGs, 8 with both periods) is a small group. Among the ten groups of
20 or more DPGs, health has the largest fall.

In [38]:
rank = t9[t9.n_both > 0].sort_values("core_change")                  # 16 groups with data in both periods; SDG 14 has none
print(rank[["dpgs", "n_both", "core_change"]].assign(core_change=lambda x: (x.core_change * 100).round(1)).to_string())
print("groups with no DPG in both periods:", t9[t9.n_both == 0].index.tolist())
big10 = t9[t9.dpgs >= 20]
health_lowest = bool(big10.core_change.idxmin() == 3) and len(big10) == 10
vals = []
for _, r_ in rank.iterrows():                                         # change (%), DPGs, DPGs with both periods, in ranking order
    vals += [r_.core_change * 100, r_.dpgs, r_.n_both]
vals += [0]                                                            # SDG 14: n_both = 0
record_result("N125", "SDG groups ranked by change in core contributors: change %, DPGs, DPGs with both periods; SDG14 has none", vals, ok=health_lowest)

     dpgs  n_both  core_change
sdg                           
2      12       8        -34.0
3      60      54        -26.3
7       4       4        -22.6
12      9       6        -13.3
13     20      17        -13.1
4      32      30        -10.2
17     27      26         -9.0
6       9       8         -6.4
5      15      15         -4.6
11     24      22         -4.3
15      7       4         -1.0
9      52      43         -0.9
16     44      39          0.5
10     24      23          6.8
8      24      20          6.8
1      20      17         19.9
groups with no DPG in both periods: [14]


### Claim N119: it is not just the funding-exposed projects
Health recomputed without the SDG 3 DPGs flagged as exposed to US aid (primary definition, notebook 05), against all 144 DPGs
with data in both periods and the 90 outside health.

In [39]:
flg = pd.read_csv(DATA / "funding_flags.csv")
exposed_ids = set(flg[flg.exposed == 1].dpgid)
no_exp = grp([i for i in h3 if i not in exposed_ids])
all_both = grp(list(both)); non_h = grp([i for i in both if i not in h3])
print("exposed DPGs that are in SDG3:", [nm.get(i) for i in h3 if i in exposed_ids], f"({len([i for i in exposed_ids if i in h3])} of {len(exposed_ids)})")
print(f"SDG3 minus exposed: n={no_exp['n']} {no_exp['a']:.1f}->{no_exp['b']:.1f} ({no_exp['chg']:.1f}%) | all {all_both['n']}: {all_both['chg']:.1f}% | outside health {non_h['n']}: {non_h['chg']:.1f}%")
record_result("N119", "SDG3 change without the exposed DPGs, %; all DPGs with both periods, % and n; outside health, % and n; exposed DPGs in SDG3, of exposed",
         [no_exp["chg"], all_both["chg"], all_both["n"], non_h["chg"], non_h["n"], len([i for i in exposed_ids if i in h3]), len(exposed_ids)])

exposed DPGs that are in SDG3: ['Community Health Toolkit', 'JemPI', 'Harmony', 'CommCare', None] (5 of 12)
SDG3 minus exposed: n=50 222.2->173.3 (-22.0%) | all 144: -9.1% | outside health 90: -2.5%


### Claim N120: health's thinning begins in early 2024, a year before the freeze
On the balanced panel (100 DPGs, 38 in health) the sum of core contributors per quarter, indexed to its 2023 average
(2023Q1-2023Q4 = 1.00), for the health DPGs and for the other DPGs.

In [40]:
qq = d[d.dpgid.isin(bal_ids) & (d.quarter >= "2021Q1")]            # balanced panel, complete quarters
s3 = qq[qq.dpgid.isin(h3)].groupby("quarter").core_team_dpg.sum()
so = qq[~qq.dpgid.isin(h3)].groupby("quarter").core_team_dpg.sum()
i3 = s3 / s3.loc["2023Q1":"2023Q4"].mean(); io_ = so / so.loc["2023Q1":"2023Q4"].mean()
print(pd.DataFrame({"health": s3, "other": so, "health_idx": i3.round(2), "other_idx": io_.round(2)}).loc["2023Q1":].to_string())
print("DPGs in health / outside health (balanced):", qq[qq.dpgid.isin(h3)].dpgid.nunique(), qq[~qq.dpgid.isin(h3)].dpgid.nunique())
oth = io_.loc["2023Q4":]
record_result("N120", "health index 2023Q4, 2024Q1, 2024Q3, 2024Q4, 2025Q1, 2026Q3; other DPGs' index range from 2023Q4",
         [i3["2023Q4"], i3["2024Q1"], i3["2024Q3"], i3["2024Q4"], i3["2025Q1"], i3["2026Q3"], oth.min(), oth.max()])

         health  other  health_idx  other_idx
quarter                                      
2023Q1      242    632        1.01       1.00
2023Q2      252    631        1.05       0.99
2023Q3      246    627        1.02       0.99
2023Q4      223    650        0.93       1.02
2024Q1      210    638        0.87       1.00
2024Q2      228    626        0.95       0.99
2024Q3      205    615        0.85       0.97
2024Q4      194    663        0.81       1.04
2025Q1      187    699        0.78       1.10
2025Q2      177    627        0.74       0.99
2025Q3      179    632        0.74       1.00
2025Q4      180    640        0.75       1.01
2026Q1      187    659        0.78       1.04
2026Q2      186    650        0.77       1.02
2026Q3      172    641        0.71       1.01
DPGs in health / outside health (balanced): 38 62


### Claim N121: health is one of 17 groups, and a random set of 54 DPGs rarely falls as far
17 SDG groups were looked at. Among the four largest groups, health has the lowest share building AI features. A **permutation
check** (5,000 random sets of 54 DPGs, the number of health DPGs with data in both periods) asks how often a random set's change is
as large a fall as -24.5%. Seed 0, drawn after the bootstrap above in the same generator (the cell replays the bootstrap draws so
that it can be re-run on its own). Random draws: the same seed and numpy version give the same share; with another numpy
version expect 3.2% give or take a few tenths.

In [41]:
rng1 = np.random.default_rng(0)                                  # same FIXED seed as the bootstrap ...
_ = sdg3_bootstrap(rng1)                                         # ... replay its 5,000 draws so the generator is in the same state
av, bv = c23[list(both)].to_numpy(), c26[list(both)].to_numpy()  # all DPGs with both periods
k_ = len(in3)                                                    # 54: size of the health group
perm = np.array([bv[idx].sum() / av[idx].sum() - 1 for idx in (rng1.choice(len(av), k_, replace=False) for _ in range(5000))])
share_perm = (perm <= -0.245).mean() * 100
print(f"{len(t9)} SDG groups | permutation: {share_perm:.1f}% of random sets of {k_} fall by 24.5% or more (5th percentile {np.percentile(perm, 5) * 100:.1f}%)")
big = t9[t9.dpgs > 30]
print(big[["dpgs", "ai_share"]].to_string())
record_result("N121", "number of SDG groups; AI share (%) and SDG number for SDG3, 9, 16, 4; permutation share (%)",
         [len(t9), 3, t9.loc[3, "ai_share"] * 100, 9, t9.loc[9, "ai_share"] * 100, 16, t9.loc[16, "ai_share"] * 100, 4, t9.loc[4, "ai_share"] * 100, share_perm])

17 SDG groups | permutation: 3.2% of random sets of 54 fall by 24.5% or more (5th percentile -22.6%)


     dpgs  ai_share
sdg                
3      60     0.233
9      52     0.423
16     44     0.523
4      32     0.438


## 9. Summary of the numbers reproduced here

In [42]:
print_results()

N033  DPGs in the balanced panel
      -> 100
N048  core contributors per quarter 2021 / 2023 / 2025 / 2026
      -> 805.25, 875.75, 830.25, 831.667
N049  share of DPG-quarters with 0 core contributors (%), 2021 and 2026
      -> 12.5, 24
N050  median core contributors per DPG-quarter, 2021 and 2026
      -> 5, 4
N039  merged PRs per core contributor 2021 / 2024 / 2025 / 2026 and rise 2021->2026 (%)
      -> 8.75473, 10.7674, 11.0346, 14.7595, 68.589
N040  output A per core contributor 2021 / 2024 / 2025 / 2026
      -> 7.3946, 6.58834, 5.72358, 6.17675
N051  2018-23 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026
      -> 52, 3, 1.5, 18.75, 35.2564
N052  pre-2018 group: n DPGs; median core 2021 and 2026; share with none (%) 2021 and 2026
      -> 48, 7, 6, 5.72917, 11.8056
N034  feature-labelled merged PRs 2021 and 2026
      -> 3121, 5247
N035  feature share of merged PRs (%), 2021 and 2026
      -> 11.0678, 14.2485
N036  minor+major releases 2021 and 2026